# Adicionando funções externas a API da OpenAI

Um grande salto de possibilidades de utilizações únicas da LLMs ocorrreu quando a OpenAI lançou o function calling. Essa ferramenta permite adicionarmos manualmente funções externas ao modelo que ele, dependendo da situação, poderá utilizar para obter novas informações ou atuar em diversos escopos. Vamos fazer uma breve revisão de como utilizamos funções externas na api da OpenAI, este assunto é explorado mais afundo no curso de Explorando a API da OpenAI. Na próxima aula, mostraremos como o framework langChain facilita a utilização das funções externas.

## importando a biblioteca



In [1]:
print('hello world')

hello world


In [9]:
# Importar Bibliotecas Necessárias
import os
from dotenv import load_dotenv
import requests
from langchain_community.llms import Ollama

# Carregar variáveis de ambiente
load_dotenv()

print("✅ Bibliotecas importadas com sucesso!!!! ")

✅ Bibliotecas importadas com sucesso!!!! 


In [10]:
# Verificar conexão com Ollama
OLLAMA_BASE_URL = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
    if response.status_code == 200:
        modelos = response.json()
        print("✅ Conexão com Ollama estabelecida!")
        print(f"\n📦 Modelos disponíveis:")
        for modelo in modelos.get('models', []):
            print(f"  - {modelo['name']}")
    else:
        print("❌ Erro ao conectar com Ollama")
except requests.exceptions.ConnectionError:
    print("❌ Não foi possível conectar ao Ollama. Certifique-se que está rodando em Docker")

✅ Conexão com Ollama estabelecida!

📦 Modelos disponíveis:
  - qwen3.5:9b
  - llama3.1:8b-instruct-q4_K_M
  - nomic-embed-text:latest
  - llama3.2:3b
  - nomic-embed-text:v1.5
  - qwen3:4b
  - olmo-3:7b
  - llama3.2:1b
  - llama3.1:8b
  - llama3.2:latest


## Importações iniciais

In [11]:
import json
# import openai
import ollama
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

client = ollama.Client(host=OLLAMA_BASE_URL)

## Criando função que será adicionada ao modelo

Utilizaremos uma função simples que simula uma api de tempo, que retorna a temperatura de um determinado local. Lembrando que modelos de llm são treinados com dados históricos, portanto, não possuem informações atuais. A única forma de eles entenderem o que está ocorrendo neste instante é passando informações pra eles através de prompts ou de funções externas.

In [12]:
def obter_horario_atual(local):
    import datetime
    horario_atual = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if "são paulo" in local.lower():
        return json.dumps({"local": "São Paulo","horario": horario_atual})
    elif "porto alegre" in local.lower():
        return json.dumps({"local": "Porto Alegre","horario": horario_atual })
    else :
        return json.dumps(
            {"local": local, "horario": "unknown"}
        )

In [13]:
def obter_temperatura_atual(local, unidade="celsius"):
    hora = horario = json.loads(obter_horario_atual("São Paulo"))["horario"]
    if "são paulo" in local.lower():
        return json.dumps(
            {"local": "São Paulo", "temperatura": "32", "unidade": unidade, "horario": hora }
            )
    elif "Porto Alegre" in local.lower():
        return json.dumps(
            {"local": "Porto Alegre", "temperatura": "25", "unidade": unidade, "horario": hora}
            )
    else:
        return json.dumps(
            {"local": local, "temperatura": "unknown"}
            )

In [14]:
obter_temperatura_atual('são paulo')

'{"local": "S\\u00e3o Paulo", "temperatura": "32", "unidade": "celsius", "horario": "2026-03-27 11:00:59"}'

## Criando descrição da função

Através dessa descrição o modelo entenderá o que a função faz e como ela pode ser utilizada

In [15]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "obter_temperatura_atual",
            "description": "Obtém a temperatura atual em uma dada cidade e horário",
            "parameters": {
                "type": "object",
                "properties": {
                    "local": {
                        "type": "string",
                        "description": "O nome da cidade. Ex: São Paulo",
                    },
                    "unidade": {
                        "type": "string", 
                        "enum": ["celsius", "fahrenheit"]
                    },
                    "horario": {
                        "type": "string",
                        "description": "O horário para o qual se deseja obter a temperatura. Ex: 2024-06-01 14:00:00"
                    }
                },
                "required": ["local" ],
            },
        },
    }
    ]

## Chamando o modelo com a nova ferramenta

Para chamar o modelo com a ferramenta criada, basta passar o argumento tools com uma lista de ferramentas.

In [21]:
mensagens = [
    {'role': 'user', 'content': 'Qual é temperatura em Porto Alegre agora?'}
]

resposta = client.chat(
    model='qwen3.5:9b',
    messages=mensagens,
    tools=tools,
)
resposta

ChatResponse(model='qwen3.5:9b', created_at='2026-03-27T14:09:10.217214Z', done=True, done_reason='stop', total_duration=118260756800, load_duration=11163121800, prompt_eval_count=370, prompt_eval_duration=30453882200, eval_count=281, eval_duration=76232700700, message=Message(role='assistant', content='', thinking='The user is asking about the current temperature in Porto Alegre (Porto Alegre). I need to use the obter_temperatura_atual function to get this information.\n\nLooking at the function parameters:\n- local: The city name (Porto Alegre)\n- horario: The time for which to get the temperature (the user asked for "now", so I should use current time or a reasonable current time)\n- unidade: The temperature unit (celsius or fahrenheit)\n\nSince the user asked in Portuguese, they likely want Celsius. For the time, since they asked for "now", I should use the current time. However, I don\'t have access to the exact current time, so I\'ll use a reasonable recent time format. Let me us

### Analisando a resposta

Podemos perceber que o conteúdo da resposta veio vazio, pois para a pergunta "Qual é a temperatura em Porto Alegre?" ele necessitará chamar a função antes.

In [22]:
mensagem = resposta['message']

In [23]:
mensagem.content

''

In [24]:
mensagem.tool_calls

[ToolCall(function=Function(name='obter_temperatura_atual', arguments={'local': 'Porto Alegre', 'unidade': 'celsius', 'horario': '2024-06-01 14:00:00'}))]

In [25]:
tools_call = mensagem.tool_calls[0]
print(tools_call.function.name)
print(tools_call.function.arguments)

obter_temperatura_atual
{'local': 'Porto Alegre', 'unidade': 'celsius', 'horario': '2024-06-01 14:00:00'}


### Adicionando resultado da função as mensagens

In [26]:
observacao = obter_temperatura_atual(**json.loads(tools_call.function.arguments))
observacao

TypeError: the JSON object must be str, bytes or bytearray, not dict

### Chamando novamente o modelo

In [27]:
mensagens.append(mensagem)
mensagens

[{'role': 'user', 'content': 'Qual é temperatura em Porto Alegre agora?'},
 Message(role='assistant', content='', thinking='The user is asking about the current temperature in Porto Alegre (Porto Alegre). I need to use the obter_temperatura_atual function to get this information.\n\nLooking at the function parameters:\n- local: The city name (Porto Alegre)\n- horario: The time for which to get the temperature (the user asked for "now", so I should use current time or a reasonable current time)\n- unidade: The temperature unit (celsius or fahrenheit)\n\nSince the user asked in Portuguese, they likely want Celsius. For the time, since they asked for "now", I should use the current time. However, I don\'t have access to the exact current time, so I\'ll use a reasonable recent time format. Let me use today\'s date with current hour - I\'ll use a format that makes sense.\n\nActually, I should use a more generic current time since I don\'t know the exact current time. Let me use a reasonable

In [28]:
mensagens.append({
    'tool_call_id': tools_call.id,
    'role': 'tool',
    'name': tools_call.function.name,
    'content': observacao
})
mensagens

AttributeError: 'ToolCall' object has no attribute 'id'

In [29]:
mensagens.append(mensagem)

mensagens

mensagens.append({

    'tool_call_id': tools_call.id,

    'role': 'tool',

    'name': tools_call.function.name,

    'content': observacao

})

mensagens

resposta = client.chat(

    model='qwen3:4b',

    messages=mensagens,

    tools=tools,

)

resposta

AttributeError: 'ToolCall' object has no attribute 'id'

In [30]:
resposta['message']['content']

''

## Explorando diferentes perguntas e o parâmetro tool_choice

Através do parâmetro tool_choice é possível forçar o modelo a sempre utilizar uma tool. Vamos ver como ele se comporta para diferentes perguntas modificando o parâmetro.

### Parâmetro "auto"

Assim o modelo define automaticamente se é necessária a utilização de uma função ou não

In [ ]:
mensagens = [
    {'role': 'user', 'content': 'Qual é temperatura em Porto Alegre agora?'}
]
resposta = client.chat(
    model='qwen3:4b',
    messages=mensagens,
    tools=tools,
)
mensagem = resposta['message']
print('Conteúdo:', mensagem.content)
print('Tools:', mensagem.tool_calls)

Conteúdo: None
Tools: [ChatCompletionMessageToolCall(id='call_hIB0t5IodL40cARvTUw3LDrT', function=Function(arguments='{"local":"Porto Alegre","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')]


In [ ]:
mensagens = [
    {'role': 'user', 'content': 'Olá'}
]
resposta = client.chat(
    model='qwen3:4b',
    messages=mensagens,
    tools=tools,
)
mensagem = resposta['message']
print('Conteúdo:', mensagem.content)
print('Tools:', mensagem.tool_calls)

Conteúdo: Olá! Como posso te ajudar hoje?
Tools: None


### Parâmetro "none"

Com o parâmetro "none", o modelo não vai utilizar funções.

In [ ]:
mensagens = [

    {'role': 'user', 'content': 'Qual a temperatura em Porto Alegre?'}

]

resposta = client.chat(

    model='qwen3:4b',

    messages=mensagens,

    tools=tools,

)

mensagem = resposta['message']

print('Conteúdo:', mensagem.content)

print('Tools:', mensagem.tool_calls)

Conteúdo: Vou verificar a temperatura atual em Porto Alegre para você. Apenas um momento, por favor.
Tools: None


In [ ]:
mensagens = [

    {'role': 'user', 'content': 'Olá'}

]

resposta = client.chat(

    model='qwen3:4b',

    messages=mensagens,

    tools=tools,

)

mensagem = resposta['message']

print('Conteúdo:', mensagem.content)

print('Tools:', mensagem.tool_calls)

Conteúdo: Olá! Como posso te ajudar hoje?
Tools: None


### Parâmetro "function"

Podemos fazer o modelo rodar obrigatoriamente a função, passando dentro de um dicionário a função que o modelo deve rodar.

In [ ]:
mensagens = [

    {'role': 'user', 'content': 'Qual a temperatura em Porto Alegre?'}

]

resposta = client.chat(

    model='qwen3:4b',

    messages=mensagens,

    tools=tools,

)

mensagem = resposta['message']

print('Conteúdo:', mensagem.content)

print('Tools:', mensagem.tool_calls)

Conteúdo: None
Tools: [ChatCompletionMessageToolCall(id='call_yZBuro6t06vP6pxAbpskf4HJ', function=Function(arguments='{"local":"Porto Alegre"}', name='obter_temperatura_atual'), type='function')]


In [ ]:
mensagens = [

    {'role': 'user', 'content': 'Olá'}

]

resposta = client.chat(

    model='qwen3:4b',

    messages=mensagens,

    tools=tools,

)

mensagem = resposta['message']

print('Conteúdo:', mensagem.content)

print('Tools:', mensagem.tool_calls)

Conteúdo: None
Tools: [ChatCompletionMessageToolCall(id='call_gGOsnHojNAg9seWLHG2qSCmy', function=Function(arguments='{"local":"São Paulo","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')]
